# Day 14 — LLM Apps: Review Summaries & Customer Support Chatbot
**RetailMind AI increment**

## What you will build
1. Summarise customer reviews with a Hugging Face Transformers summarization pipeline.
2. Connect to an OpenAI LLM API for customer-support answers.
3. Keep chat history during the current notebook session.
4. Ground answers using simple product information and policy text.

**Beginner-friendly approach:** This notebook starts with a small built-in review sample, so it runs without downloading the full Amazon dataset. You can optionally load your own review CSV later.

**Important:** The summarization model downloads the first time and may take several minutes. The chatbot requires an OpenAI API key. Never paste your key into a notebook cell or commit it to GitHub.

## 1. Install the required libraries
Run this cell once in your active Jupyter kernel. Restart the kernel if Jupyter asks you to.

In [ ]:
%pip install -U transformers torch openai

## 2. Imports and basic setup

In [1]:
import os
import csv
from pathlib import Path
from getpass import getpass

print("Day 14 setup started.")

Day 14 setup started.


## 3. Prepare review data

We use a small example first to make sure the notebook works. Later, you can load your own CSV file.

If you have a CSV, put it in `data/` and set `CSV_PATH` below. The CSV should contain a column named `review` or `content` (case-insensitive). If you do not have one, leave `CSV_PATH` as `None`.

In [2]:
# Small sample so the notebook can be tested immediately.
reviews = [
    "The shoes are comfortable and lightweight. I can wear them all day.",
    "Good quality for the price. The design looks nice and the size fits well.",
    "The sole started coming off after two weeks. I am disappointed with the durability.",
    "Delivery was late, but the product itself is comfortable.",
    "The material feels soft and the shoes look exactly like the pictures.",
    "The size runs smaller than expected. I recommend ordering one size up.",
    "Excellent value for money. The stitching and finish are good.",
    "The box arrived damaged and delivery took longer than promised.",
    "Very comfortable for walking, but the colour is slightly different from the image.",
    "Poor durability. The shoe became loose after a few days."
]

CSV_PATH = None  # Example: Path("../data/amazon_reviews.csv")

def load_reviews_from_csv(csv_path):
    csv_path = Path(csv_path)
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV file not found: {csv_path}")

    with csv_path.open("r", encoding="utf-8-sig", newline="") as file:
        reader = csv.DictReader(file)
        if not reader.fieldnames:
            raise ValueError("The CSV file has no header row.")

        columns = {name.lower().strip(): name for name in reader.fieldnames}
        review_column = columns.get("review") or columns.get("content") or columns.get("text")
        if review_column is None:
            raise ValueError("CSV needs a 'review', 'content', or 'text' column.")

        loaded = [
            row.get(review_column, "").strip()
            for row in reader
            if row.get(review_column, "").strip()
        ]
    return loaded

if CSV_PATH is not None:
    reviews = load_reviews_from_csv(CSV_PATH)

print("Number of reviews:", len(reviews))
print("\nFirst review:", reviews[0])

Number of reviews: 10

First review: The shoes are comfortable and lightweight. I can wear them all day.


## 4. Load the Hugging Face summarization pipeline

We use `sshleifer/distilbart-cnn-12-6`, a smaller summarization model than the full BART model. The first run downloads model files from Hugging Face.

If your computer has no GPU, the pipeline will run on CPU; it may be slower.

In [5]:
import os

# Force Transformers to use PyTorch
os.environ["USE_TF"] = "0"

import torch
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1

print("Using:", "GPU" if device == 0 else "CPU")

summarizer = pipeline(
    "summarization",
    model="sshleifer/distilbart-cnn-12-6",
    framework="pt",
    device=device
)

print("Summarization model loaded successfully!")

Using: CPU


pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

c:\Users\ANSHU SHAW\OneDrive\Desktop\RETAILMIND_AI\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ANSHU SHAW\.cache\huggingface\hub\models--sshleifer--distilbart-cnn-12-6. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 9563d044-048e-4357-9ae0-18ce2e9eb5fe)')' thrown while requesting HEAD https://huggingface.co/api/resolve-cache/models/sshleifer/distilbart-cnn-12-6/a4f8f3ea906ed274767e9906dbaede7531d660ff/merges.txt
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: e05ee0eb-34bf-4fd2-90d1-6137d73da568)')' thrown while requesting HEAD https://huggingface.co/api/resolve-cache/models/sshleifer/distilbart-cnn-12-6/a4f8f3ea906ed274767e9906dbaede7531d660ff/merges.txt
Retrying in 2s [Retry 2/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 0224571d-e938-410e-80a7-9fec6c2da7bd)')' thrown while requesting HEAD https://huggingface.co/api/resolve-cache/models/sshleifer/distilbart-cnn-12-6/a4f8f3ea906ed274767e9

merges.txt: 0.00B [00:00, ?B/s]

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 70e4079c-2187-4c89-ab4f-17485ac97638)')' thrown while requesting HEAD https://huggingface.co/sshleifer/distilbart-cnn-12-6/resolve/main/tokenizer.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /sshleifer/distilbart-cnn-12-6/resolve/main/tokenizer.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: d03f1f73-630c-4451-bf40-21d5d0031a40)')' thrown while requesting HEAD https://huggingface.co/sshleifer/distilbart-cnn-12-6/resolve/main/tokenizer.json
Retrying in 2s [Retry 2/5].
Device set to use cpu


Summarization model loaded successfully!


In [4]:
import transformers
print(transformers.__version__)

4.57.3


## 5. Summarise reviews for one product

We combine reviews into manageable groups and summarise each group. Then we create a short overall summary from those group summaries.

In [6]:
def split_into_chunks(text_list, max_words=450):
    chunks = []
    current_chunk = []
    current_words = 0

    for review in text_list:
        word_count = len(review.split())

        if current_chunk and current_words + word_count > max_words:
            chunks.append(" ".join(current_chunk))
            current_chunk = []
            current_words = 0

        current_chunk.append(review)
        current_words += word_count

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks


def summarise_reviews(review_list):
    if not review_list:
        return "No reviews were provided."

    chunks = split_into_chunks(review_list)
    partial_summaries = []

    for number, chunk in enumerate(chunks, start=1):
        result = summarizer(
            chunk,
            max_length=100,
            min_length=25,
            do_sample=False,
            truncation=True
        )
        partial_summaries.append(result[0]["summary_text"])
        print(f"Summarised group {number}/{len(chunks)}")

    if len(partial_summaries) == 1:
        return partial_summaries[0]

    combined = " ".join(partial_summaries)
    final_result = summarizer(
        combined,
        max_length=120,
        min_length=35,
        do_sample=False,
        truncation=True
    )
    return final_result[0]["summary_text"]


review_summary = summarise_reviews(reviews)
print("\nPRODUCT REVIEW SUMMARY")
print("----------------------")
print(review_summary)

Summarised group 1/1

PRODUCT REVIEW SUMMARY
----------------------
 The box arrived damaged and delivery took longer than promised . The material feels soft and the shoes look exactly like the pictures . The colour is slightly different from the image . The shoes are comfortable and lightweight .


## 6. Show simple review themes

This is a basic keyword-based view to help inspect the reviews. It is not a machine-learning sentiment classifier; the LLM summary above is the main summarization output.

In [7]:
positive_words = ["comfortable", "good", "excellent", "soft", "nice", "value", "lightweight"]
negative_words = ["poor", "late", "damaged", "disappointed", "smaller", "loose", "different"]

positive_mentions = sum(
    1 for review in reviews
    if any(word in review.lower() for word in positive_words)
)
negative_mentions = sum(
    1 for review in reviews
    if any(word in review.lower() for word in negative_words)
)

print("Reviews:", len(reviews))
print("Reviews mentioning positive terms:", positive_mentions)
print("Reviews mentioning negative terms:", negative_mentions)

Reviews: 10
Reviews mentioning positive terms: 6
Reviews mentioning negative terms: 6


## 7. Add product information and store policies

Replace this example text with the actual product metadata and official RetailMind store policies. The chatbot must not invent policies.

In [8]:
PRODUCT_CONTEXT = '''
Product: Everyday Walking Shoes
Category: Footwear
Material: Breathable mesh upper with a rubber sole
Highlights: Lightweight design, cushioned footbed, everyday walking use
'''

POLICY_CONTEXT = '''
Returns: Eligible unused products can be returned within 10 days of delivery.
Refunds: Refunds are processed after the returned item passes inspection.
Delivery: Delivery estimates are shown at checkout and may vary by location.
Support: Customers should provide their order ID when asking about a specific order.
'''

KNOWLEDGE_CONTEXT = PRODUCT_CONTEXT + "\n" + POLICY_CONTEXT
print(KNOWLEDGE_CONTEXT)


Product: Everyday Walking Shoes
Category: Footwear
Material: Breathable mesh upper with a rubber sole
Highlights: Lightweight design, cushioned footbed, everyday walking use


Returns: Eligible unused products can be returned within 10 days of delivery.
Refunds: Refunds are processed after the returned item passes inspection.
Delivery: Delivery estimates are shown at checkout and may vary by location.
Support: Customers should provide their order ID when asking about a specific order.



## 8. Set up the Groq API key safely

Create an API key in your OpenAI account. This cell asks for it privately if the `GROQ_API_KEY` environment variable is not already set.

The key is not written into this notebook. API usage may incur charges depending on your account and model access.

In [ ]:
import os
from groq import Groq
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found. Check your .env file.")

client = Groq(api_key=GROQ_API_KEY)

Groq client connected successfully!


## 9. Create a simple support chatbot

The chatbot sends the product/policy text and the recent conversation history with each question. This is a simple context-based chatbot, not a full retrieval-augmented generation (RAG) system.

In [16]:
def ask_retailmind(question, model="openai/gpt-oss-120b"):

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT}
    ]

    messages.extend(chat_history[-10:])
    messages.append({"role": "user", "content": question})

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.2,
        max_tokens=500
    )

    answer = response.choices[0].message.content

    chat_history.append({
        "role": "user",
        "content": question
    })

    chat_history.append({
        "role": "assistant",
        "content": answer
    })

    return answer

## 10. Test the chatbot

In [17]:
question = "What is the return policy?"
answer = ask_retailmind(question)

print("Customer:", question)
print("RetailMind AI:", answer)

Customer: What is the return policy?
RetailMind AI: Sure! If the product is unused, you can return it within **10 days of delivery**. Once we receive the item and it passes inspection, we’ll process your refund. If you have any other questions, feel free to ask!


## 11. Continue the conversation

Try a follow-up question. The previous messages are retained in `chat_history` during this notebook session.

In [18]:
question = "What happens after I return the product?"
answer = ask_retailmind(question)

print("Customer:", question)
print("RetailMind AI:", answer)

print("\nConversation message count:", len(chat_history))

Customer: What happens after I return the product?
RetailMind AI: After you send the product back, our team will inspect the item to confirm it’s unused and in its original condition. Once it passes inspection, we’ll process your refund. The refund will be issued using the same payment method you used for the purchase. If you have any other questions or need further assistance, just let me know!

Conversation message count: 4


## 12. Ask your own questions

Run this cell and enter a question. Run it again for another question. Stop by interrupting the cell.

In [19]:
question = input("You: ").strip()
if question.lower() in {"exit", "quit"}:
    print("Chat ended.")
else:
    print("RetailMind AI:", ask_retailmind(question))

RetailMind AI: I’m here to help! Let me know if you have any questions about our products, orders, returns, or anything else you need assistance with.


## 13. Reset conversation history

Run this cell when you want to start a fresh conversation.

In [20]:
chat_history.clear()
print("Conversation history cleared.")

Conversation history cleared.


## 14. Save the review summary

This saves the generated summary as a text file inside `reports/day14/`. It creates the folder if needed.

In [21]:
report_dir = Path("../reports/day14")
report_dir.mkdir(parents=True, exist_ok=True)

summary_path = report_dir / "day14_review_summary.txt"
summary_path.write_text(
    "RetailMind AI — Day 14 Review Summary\n\n" + review_summary,
    encoding="utf-8"
)

print("Saved summary to:", summary_path.resolve())

Saved summary to: C:\Users\ANSHU SHAW\OneDrive\Desktop\RETAILMIND_AI\reports\day14\day14_review_summary.txt


## 15. Day 14 validation

These checks confirm that the key outputs and functions exist.

In [ ]:
assert len(reviews) > 0, "No reviews available."
assert isinstance(review_summary, str) and len(review_summary) > 0
assert callable(ask_retailmind)
assert isinstance(chat_history, list)
assert summary_path.exists()

print("All Day 14 local validation checks passed.")

## Day 14 Summary

In this increment, you implemented:

- Review summarization using Hugging Face Transformers.
- Review grouping for handling multiple reviews.
- OpenAI API integration with a key entered securely at runtime.
- A customer-support chatbot grounded in product and policy text.
- Conversation history for follow-up questions.
- A saved review summary and basic validation.

### Before pushing to GitHub

1. Save this notebook as `notebooks/Day_14_LLM_Apps.ipynb`.
2. Keep your API key out of the notebook and repository.
3. Do not commit `.env` or any file containing secrets.
4. Add the Day 14 notebook and generated report to Git only if you want to track the report.

### Suggested Git commands

```bash
git status
git add notebooks/Day_14_LLM_Apps.ipynb
git add reports/day14/day14_review_summary.txt
git commit -m "Day 14: LLM review summarization and support chatbot"
git push origin main
```

**Note:** If you use a real Amazon review CSV, check its terms and dataset licence. The built-in sample is only for testing the notebook.